# Investigacion por hostname o direccion MAC

## Objetivo

Seguir un dispositivo aunque cambie de IP, usando identidad DHCP y actividad DNS correlacionada.

## Entradas esperadas

- `TargetHost` o `TargetMac`.
- `Lookback`: ventana de tiempo.

## Requisitos

- Acceso al area de trabajo de Microsoft Sentinel.
- Funciones KQL publicadas: `fn_Normalize_Windows_DHCP`, `fn_Normalize_Windows_DNS`, `fn_Correlate_DHCP_DNS`.
- Paquetes Python sugeridos: `msticpy`, `pandas`, `matplotlib`, `plotly`, `networkx` segun el notebook.

## Secciones

1. IPs usadas.
2. Scopes DHCP.
3. Actividad DNS por IP.
4. Cambios anormales.


In [ ]:
# Configuracion general - ajustar antes de ejecutar
workspace_id = "REEMPLAZAR_CON_WORKSPACE_ID"
tenant_id = "REEMPLAZAR_CON_TENANT_ID"

# Conexion sugerida con MSTICPy
# import msticpy as mp
# mp.init_notebook(namespace=globals())
# qry_prov = mp.QueryProvider("MSSentinel")
# qry_prov.connect(workspace=workspace_id, tenant_id=tenant_id)


In [ ]:
query_identity = """
let TargetHost = tolower("REEMPLAZAR_CON_HOSTNAME");
let TargetMac = toupper("REEMPLAZAR_CON_MAC");
let Lookback = 7d;
fn_Normalize_Windows_DHCP(Lookback)
| where tolower(HostName) == TargetHost or toupper(ClientMac) == TargetMac
| summarize FirstSeen=min(TimeGenerated), LastSeen=max(TimeGenerated), Ips=make_set(ClientIp, 50), Scopes=make_set(ScopeId, 20), DhcpServers=make_set(DeviceName, 20) by HostName, ClientMac
"""
# identity_df = qry_prov.exec_query(query_identity)
print(query_identity)


In [ ]:
query_dns_by_device = """
let TargetHost = tolower("REEMPLAZAR_CON_HOSTNAME");
let TargetMac = toupper("REEMPLAZAR_CON_MAC");
let Lookback = 7d;
fn_Correlate_DHCP_DNS(Lookback)
| where tolower(HostName) == TargetHost or toupper(ClientMac) == TargetMac
| summarize Queries=count(), DistinctDomains=dcount(QueryRootDomain), SampleDomains=make_set(QueryRootDomain, 20) by ClientIp, HostName, ClientMac
| order by Queries desc
"""
# dns_by_device_df = qry_prov.exec_query(query_dns_by_device)
print(query_dns_by_device)


## Resumen para incidente

Documentar aqui:

- Hallazgos principales.
- Entidades relevantes: IP, hostname, direccion MAC, dominio.
- Evidencia KQL usada.
- Recomendacion: cerrar, monitorear, escalar o contener.
